## Import Functions

In [ ]:
from __future__ import annotations

# === Standard Library ===
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Iterable, Optional, Type, Union

# === Third-Party Libraries ===
import numpy as np
from numba.np.math.mathimpl import f32_as_int32
from numpy.typing import NDArray

import torch
from torch import Tensor

import whisper
import soundfile as sf

import yaml
import phonemizer
from phonemizer.backend import EspeakBackend

from nltk.cluster import cosine_distance
from nltk.tokenize import word_tokenize

from sentence_transformers import SentenceTransformer
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import math

import IPython.display as ipd

# === Pymoo Optimization Framework ===
from _pymoo_optimizer import PymooOptimizer

from pymoo.algorithms.base.genetic import GeneticAlgorithm
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.evaluator import Evaluator
from pymoo.core.population import Population
from pymoo.core.problem import Problem
from pymoo.core.termination import NoTermination
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.problems.static import StaticProblem

# === StyleTTS2 Local Modules ===
from models import *
from utils import *
from text_utils import TextCleaner
from Utils.PLBERT.util import load_plbert
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

# === Import from Files ===
from _helper import length_to_mask, addNumbersPattern, interpolateWithVector, interpolateWithScalar
from _pymoo_optimizer import PymooOptimizer
from _styletts2 import StyleTTS2

%cd ..

## Create Helper Functions


In [ ]:
def text_naturalness_from_ppl(ppl, min_loss=1.0, max_loss=10.0):
    """
    ppl: perplexity from your LM
    Returns score in [0,1], where 1 = very natural/common.
    """
    loss = math.log(ppl)             # log PPL = cross-entropy-ish
    loss = max(min(loss, max_loss), min_loss)
    return 1.0 - (loss - min_loss) / (max_loss - min_loss)

## Classes

In [ ]:
class AutomaticSpeechRecognitionModel:
    def __init__(self, model_name="tiny"):
        self.model = whisper.load_model(model_name)

    def analyzeAudio(self, wav: torch.Tensor) -> tuple[dict, float]:
        result = self.model.transcribe(audio=wav)

        total_logprob = 0.0
        total_tokens = 0
        for seg in result["segments"]:
            total_logprob += seg["avg_logprob"] * len(seg["tokens"])
            total_tokens += len(seg["tokens"])

        avg_logprob = total_logprob / total_tokens if total_tokens > 0 else float("nan")
        return result, avg_logprob

In [ ]:
class TextEmbeddingModel:
    """
    Wrapper around a SentenceTransformer model for computing
    text embeddings and cosine distances.
    """

    def __init__(
        self,
        model_name: str = "sentence-transformers/all-mpnet-base-v2",
        device: Optional[str] = None,
    ):
        """
        model_name: HuggingFace model id, e.g. "sentence-transformers/all-mpnet-base-v2"
        device: "cpu", "cuda", or None to let sentence-transformers decide
        """
        self.model = SentenceTransformer(model_name, device=device)

    def computeTextEmbedding(self, text: str | list[str]) -> torch.Tensor:
        """
        Encodes a single string or list of strings.
        Returns a tensor of shape:
          - [D] for a single string
          - [N, D] for a list of N strings

        Embeddings are L2-normalized for cosine distance.
        """
        emb = self.model.encode(
            text,
            convert_to_tensor=True,
            normalize_embeddings=True,
        )
        return emb

    @staticmethod
    def cosine_distance_text_embedding(a: torch.Tensor, b: torch.Tensor) -> float:
        """
        Cosine distance in [0, ~1] for real sentence embeddings.
          0 = identical
          1 ≈ maximally different
        """
        if a.ndim == 1:
            a = a.unsqueeze(0)
        if b.ndim == 1:
            b = b.unsqueeze(0)

        b = b.to(a.device)

        cos_sim = F.cosine_similarity(a, b, dim=1)[0].item()
        return 1.0 - float(cos_sim)

    def distance_between_texts(self, text1: str, text2: str) -> float:
        """
        Convenience helper: encode two texts and return cosine distance.
        """
        e1 = self.computeTextEmbedding(text1)
        e2 = self.computeTextEmbedding(text2)
        return self.cosine_distance_text_embedding(e1, e2)

## Main Function

### Load Models

In [ ]:
tts_model = StyleTTS2()
tts_model.load_models()  # builds self.model and loads self.params
tts_model.load_checkpoints()  # puts params into self.model
tts_model.sample_diffusion()  # builds self.sampler

asr_model = AutomaticSpeechRecognitionModel("tiny")

embedding_model = TextEmbeddingModel(model_name="sentence-transformers/all-mpnet-base-v2")

perplexity_model = GPT2LMHeadModel.from_pretrained("gpt2")
perplexity_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

### Initializes Values

In [ ]:
diffusion_steps = 5
embedding_scale = 1

interpolation_percentage = 0.4 # How much of Target to be used, small interpolation_percentage means more of ground_truth (Minimization)

name_gt = "ground_truth"
text_gt = "I think the NFL is lame and boring"

name_target = "target"
text_target = "The Seattle Seahawks are the best Team in the world"

noise_gt = torch.randn(1, 1, 256).to(tts_model.device)
noise_target = torch.randn(1, 1, 256).to(tts_model.device)

text_embedding_gt = embedding_model.computeTextEmbedding(text_gt)
text_embedding_target = embedding_model.computeTextEmbedding(text_target)

h_text_gt, h_bert_raw_gt, h_bert_gt, h_text_target, h_bert_raw_target, h_bert_target, input_lengths, text_mask, style_vector_acoustic, style_vector_prosodic = tts_model.extract_mixed_embeddings(text_gt, text_target, noise_target, embedding_scale, diffusion_steps)

### Optimizer

#### Initialize Optimizer

In [ ]:
# Genetic algorithm parameters
algo_params = {
    "pop_size": 20,
}

num_generations = 1
num_objectives = 4
solution_shape = (int(input_lengths.detach().cpu().item()),)  # 16 decision variables per individual

optimizer = PymooOptimizer(
    bounds=(0, 1),
    algorithm=NSGA2,  # pass the class, not NSGA2(...)
    algo_params=algo_params,
    num_objectives=num_objectives,
    solution_shape=solution_shape,
)

#### Loop

In [ ]:
for gen in range(num_generations):

    f1_list = []
    f2_list = []
    f3_list = []
    f4_list = []

    # 1) Get current population
    population_vectors = optimizer.get_x_current()  # shape: (pop_size, *solution_shape)

    # Debugging, to verify new first individual
    print(f"=== Generation {gen} ===")
    print("First individual:", population_vectors[0][:5])

    for i, interpolation_vector in enumerate(population_vectors):

        interpolation_vector = torch.from_numpy(interpolation_vector).to(h_text_gt.device).float()

        # Interpolate Values
        h_text_mixed = interpolateWithVector(h_text_gt, h_text_target, interpolation_vector)
        h_bert_mixed = interpolateWithVector(h_bert_gt, h_bert_target, interpolation_vector)

        audio_mixed = tts_model.inference_after_interpolation(
            input_lengths,
            text_mask,
            h_bert_mixed,
            h_text_mixed,
            style_vector_acoustic,
            style_vector_prosodic
        )

        tts_model.inference_after_interpolation(input_lengths, text_mask, h_bert_gt, h_text_gt, style_vector_acoustic, style_vector_prosodic)

        asr_result, asr_confidence = asr_model.analyzeAudio(audio_mixed)
        asr_text = asr_result["text"]

        text_embedding_mixed = embedding_model.computeTextEmbedding(asr_text)

        # f1: Minimize Interpolation Vector (L1-Norm)
        f1 = float(interpolation_vector.abs().mean().item())

        # f2: Maximize Embedding Distance between ground-truth and ASR (cosine distance)
        f2 = - embedding_model.cosine_distance_text_embedding(text_embedding_gt, text_embedding_mixed)

        # f3: Minimize Embedding Distance between target and ASR (cosine distance)
        f3 = embedding_model.cosine_distance_text_embedding(text_embedding_target, text_embedding_mixed)

        # f4: Maximize ASR Confidence (Not negative, since its already a negative value)
        f4 = - asr_confidence

        # f4: Minimize Perplexity/Naturalness of ASR Text
        # perplexity_tokens = perplexity_tokenizer(asr_text, return_tensors="pt")["input_ids"]

        # with torch.no_grad():
            # outputs = perplexity_model(perplexity_tokens, labels=perplexity_tokens)
            # loss = math.exp(outputs.loss.item())

        # f4 = text_naturalness_from_ppl(loss)

        f1_list.append(f1)
        f2_list.append(f2)
        f3_list.append(f3)
        f4_list.append(f4)

    optimizer.assign_fitness(
        [
            np.array(f1_list, dtype=float),
            np.array(f2_list, dtype=float),
            np.array(f3_list, dtype=float),
            np.array(f4_list, dtype=float),
        ]
    )
    optimizer.update()


    # print(f"=== Generation {gen} ===")

    for idx, cand in enumerate(optimizer.best_candidates):
        print(f"Pareto Candidate {idx}:")
        print("Fitness vector:", cand.fitness)
        print("Solution (first 5 dims):", cand.solution[:5])
        print()


#### Inference after Interpolation

In [ ]:
print("=== Synthesis of Best Candidate After Optimization ===")

# 1) Get best candidate(s) from optimizer
best_candidates = optimizer.best_candidates

if len(best_candidates) == 0:
    raise RuntimeError("Optimizer has no best candidates. Did training run correctly?")

best = best_candidates[0]   # first Pareto-optimal candidate

# 2) Print FITNESS values
f1 = best.fitness
print("Best candidate fitness values:")
print(f"  f1 (L1 norm)                = {f1}")
# print(f"  f2 (maximize GT distance)   = {f2}")
# print(f"  f3 (minimize target dist.)  = {f3}")
# print(f"  f4 (maximize ASR conf.)     = {f4}")
print()

# 3) Extract interpolation vector
best_vector = torch.tensor(best.solution, device=tts_model.device).float()

# *** NEW: Print average value of the interpolation vector ***
avg_interp = best_vector.mean().item()
print(f"\nAverage interpolation value: {avg_interp:.6f}\n")

# 4) Create mixed embeddings using the best vector
if to_interpolate == "h_text":
    h_text_mixed_best = interpolateWithVector(h_text_gt, h_text_target, best_vector)
    h_bert_mixed_best = h_bert_gt

elif to_interpolate == "h_bert":
    h_text_mixed_best = h_text_gt
    h_bert_mixed_best = interpolateWithVector(h_bert_gt, h_bert_target, best_vector)

else:
    raise ValueError(f"Unknown interpolation target: {to_interpolate}")

# 5) Run TTS inference for:
#    - ground-truth
#    - target
#    - best mixed candidate
with torch.no_grad():
    # Ground-truth audio
    inference_gt = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_gt,
        h_text_gt,
        style_vector_acoustic,
        style_vector_prosodic,
    )
    audio_gt = tts_model.synthesizeSpeech(inference_gt)

    # Target audio
    inference_target = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_target,
        h_text_target,
        style_vector_acoustic,
        style_vector_prosodic,
    )
    audio_target = tts_model.synthesizeSpeech(inference_target)

    # Best mixed candidate audio
    inference_best = tts_model.inference_after_interpolation(
        input_lengths,
        text_mask,
        h_bert_mixed_best,
        h_text_mixed_best,
        style_vector_acoustic,
        style_vector_prosodic,
    )
    audio_best = tts_model.synthesizeSpeech(inference_best)

# 6) Run ASR on final audio and print predicted text
asr_final, conf_final = asr_model.analyzeAudio(audio_best)
predicted_text_final = asr_final["text"].strip()

print("ASR Transcription of Best Candidate:")
print(f'  "{predicted_text_final}"')
print(f"  ASR confidence (avg_logprob) = {conf_final}")
print()

# 7) Play all three audios: GT, target, and mixed best
print("=== Ground-truth Audio ===")
display(ipd.Audio(audio_gt, rate=24000))

print("=== Target Audio ===")
display(ipd.Audio(audio_target, rate=24000))

print("=== Best Mixed Candidate Audio ===")
display(ipd.Audio(audio_best, rate=24000))

print("Audio synthesis complete.")


### Write files

In [ ]:
folder_path = "outputs/audio/optimizer/"

os.makedirs(folder_path, exist_ok=True)

sf.write(folder_path + "ground_truth.wav", audio_gt, samplerate=24000)
sf.write(folder_path + "target.wav", audio_target, samplerate=24000)
sf.write(folder_path + "interpolated.wav", audio_best, samplerate=24000)